# Running Granite Time Series Ensemble R1 on GIFT-Eval

**This notebook reproduces the Granite Time Series Ensemble R1 submission to the GIFT-Eval leaderboard.**

The ensemble combines four Granite time-series foundation models:

- [Granite PatchTST-FM R1](https://huggingface.co/ibm-granite/granite-timeseries-patchtst-fm-r1)
- [Granite PatchTST-FM R2](https://huggingface.co/ibm-granite/granite-timeseries-patchtst-fm-r2)
- [Granite FlowState R1](https://huggingface.co/ibm-granite/granite-timeseries-flowstate-r1), revision `r1.1`
- [Granite TTM R3](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r3)

Their probabilistic forecasts are combined using uniform linear pooling. The published ensemble recipe is loaded from:

[ibm-granite/granite-timeseries-ensemble-r1](https://huggingface.co/ibm-granite/granite-timeseries-ensemble-r1)

This notebook:

1. Installs Granite TSFM and GIFT-Eval.
2. Downloads the published ensemble configuration and member checkpoints from Hugging Face.
3. Loads the GIFT-Eval datasets and benchmark settings.
4. Runs a small smoke test.
5. Optionally runs the complete benchmark.
6. Validates the generated `all_results.csv` and member results.

## Installation

Run this notebook from the `notebooks` directory of a cloned
[GIFT-Eval repository](https://github.com/SalesforceAIResearch/gift-eval).

Before running the notebook:

1. Follow the GIFT-Eval installation and dataset-download instructions.
2. Set `GIFT_EVAL` to the downloaded dataset directory, or set `GIFT_EVAL_PATH` in the configuration cell below.
3. Ensure the environment has access to a CUDA GPU for the full benchmark.
4. Set `HF_TOKEN` if authentication is required for Hugging Face access.

The notebook clones and installs Granite TSFM if a local checkout is not already present. Restart the kernel after changing an installed package or Granite TSFM revision.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

gift_eval_root = Path.cwd().resolve().parent
notebook_dir = Path.cwd().resolve()
if not (gift_eval_root / "pyproject.toml").is_file() or not (gift_eval_root / "src/gift_eval").is_dir():
    raise RuntimeError("Run this notebook from the GIFT-Eval notebooks directory")

granite_source = notebook_dir / "granite-tsfm"
if not os.path.exists("granite-tsfm"):
    !git clone --branch granite_tsfm_ensemble --single-branch --depth 1 https://github.com/ibm-granite/granite-tsfm.git
else:
    print("Folder 'granite-tsfm' already exists. Skipping git clone.")

%cd granite-tsfm
%pip install -q -e ".[notebooks]"
%cd ..
%pip install -q -e "..[baseline]"

sys.path.insert(0, str(granite_source.resolve()))
sys.path.insert(0, str((granite_source / "notebooks/hfdemo/gift_eval_ensemble").resolve()))

Folder 'granite-tsfm' already exists. Skipping git clone.
/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/notebooks/granite-tsfm
Note: you may need to restart the kernel to use updated packages.
/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/notebooks
Note: you may need to restart the kernel to use updated packages.


In [2]:
%cd granite-tsfm
%pip install -q -e ".[notebooks]"
%cd ..
%pip install -q -e "..[baseline]"

sys.path.insert(0, str(granite_source.resolve()))
sys.path.insert(0, str((granite_source / "notebooks/hfdemo/gift_eval_ensemble").resolve()))

/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/notebooks/granite-tsfm
Note: you may need to restart the kernel to use updated packages.
/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/notebooks
Note: you may need to restart the kernel to use updated packages.


## Configuration sources

The replication uses two complementary configurations:

- The Hugging Face repository defines the ensemble recipe: member checkpoints, model revisions, quantile levels, aggregation method, and ensemble weights.
- The Granite TSFM experiment configuration defines GIFT-Eval execution details: datasets, aliases, forecast terms, seed, output names, device selection, and resume behavior.

The Hugging Face configuration is treated as the published model definition. The local experiment configuration contains the benchmark-specific settings required by the GIFT-Eval runner.

## Data and experiment configuration

Set `GIFT_EVAL_PATH` directly in the following cell, or leave it as `None` to use the `GIFT_EVAL` environment variable.

The path must point to the directory containing the downloaded GIFT-Eval data.

The ensemble configuration is downloaded from Hugging Face using `ProbabilisticEnsembleConfig.from_pretrained()`. The GIFT-Eval execution settings are loaded from the Granite TSFM checkout.

In [3]:
import json
import os

from dotenv import load_dotenv
from tsfm_public.models.ensemble.configuration_ensemble import (
    ProbabilisticEnsembleConfig,
)

load_dotenv(gift_eval_root / ".env")

# GIFT-Eval data
GIFT_EVAL_PATH = None
if GIFT_EVAL_PATH is None:
    GIFT_EVAL_PATH = os.environ.get("GIFT_EVAL")

if not GIFT_EVAL_PATH or not Path(GIFT_EVAL_PATH).expanduser().is_dir():
    raise RuntimeError(
        "Set GIFT_EVAL_PATH or GIFT_EVAL to the downloaded dataset directory"
    )

os.environ["GIFT_EVAL"] = str(
    Path(GIFT_EVAL_PATH).expanduser().resolve()
)

# Ensemble recipe from Hugging Face
ENSEMBLE_REPO_ID = "ibm-granite/granite-timeseries-ensemble-r1"

ensemble_config = ProbabilisticEnsembleConfig.from_pretrained(
    ENSEMBLE_REPO_ID
    )
ensemble_config.validate()

# GIFT-Eval execution settings from the Granite TSFM checkout
experiment_config_path = (
    granite_source
    / "notebooks/hfdemo/gift_eval_ensemble/experiment_config.json"
)

if not experiment_config_path.is_file():
    raise FileNotFoundError(
        f"Missing GIFT-Eval experiment configuration: "
        f"{experiment_config_path}"
    )

experiment_config = json.loads(experiment_config_path.read_text())

from run_gift_eval import run_evaluation

INFO:p-81664:t-8419852160:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/ibm-granite/granite-timeseries-ensemble-r1/resolve/main/config.json "HTTP/1.1 200 OK"


In [4]:
MODEL_NAME = experiment_config["defaults"]["model_name_config"]
gift_recipe = experiment_config["configurations"][MODEL_NAME]

assert ensemble_config.aggregation_method == "linear_pool"
assert gift_recipe["ensemble"] == "probability_space_aggregation"

assert ensemble_config.quantile_levels == experiment_config["quantile_levels"]

assert len(ensemble_config.members) == len(gift_recipe["model_names"])

### Review the resolved configuration

The following cell reports:

- The model name written to the leaderboard CSV.
- The Hugging Face ensemble repository.
- The GIFT-Eval data directory.
- The output directory.
- The downloaded ensemble recipe.

Review these values before starting an evaluation.

In [5]:
SETTINGS = experiment_config["defaults"].copy()
MODEL_NAME = SETTINGS["model_name_config"]

SETTINGS.update(
    skip_processed=True,
    save_member_results=True,
)

output_root = Path(
    os.environ.get(
        "GIFT_EVAL_OUTPUT_ROOT",
        gift_eval_root / "outputs/granite_tsfm_ensemble",
    )
).expanduser().resolve()

print(f"Leaderboard model name: {MODEL_NAME}")
print(f"Ensemble repository: {ENSEMBLE_REPO_ID}")
print(f"Gift Eval Data Location: {os.environ['GIFT_EVAL']}")
print(f"Output root: {output_root}")
print(ensemble_config)

Leaderboard model name: probability-ensemble-uniform-ibm-tsfm-granite-pt
Ensemble repository: ibm-granite/granite-timeseries-ensemble-r1
Gift Eval Data Location: /Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/AlternativeTSFMAgents/timecopilot/experiments/gift-eval/data/gift-eval
Output root: /Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/outputs/granite_tsfm_ensemble
ProbabilisticEnsembleConfig {
  "aggregation_method": "linear_pool",
  "iqr_weighted_options": {
    "max_weight": 0.4,
    "temperature": 0.5
  },
  "members": [
    {
      "forecaster_type": "patchtst",
      "model_checkpoint": "ibm-granite/granite-timeseries-patchtst-fm-r1"
    },
    {
      "forecaster_type": "patchtst",
      "model_checkpoint": "ibm-granite/granite-timeseries-patchtst-fm-r2"
    },
    {
      "forecaster_type": "flowstate",
      "model_checkpoint": "ibm-granite/granite-timeseries-flowstate-r1",
      "model_revision": "r

## Record the execution environment

Library versions, repository revisions, CUDA information, and the GPU model can affect numerical reproducibility.

The following cell records the environment used for this run. Small floating-point differences may occur across GPU architectures, CUDA versions, PyTorch versions, or model-library versions.

In [6]:
import platform

import numpy as np
import pandas as pd
import torch
import transformers

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Granite revision:", subprocess.run(["git", "-C", str(granite_source), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("GIFT-Eval revision:", subprocess.run(["git", "-C", str(gift_eval_root), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

Python: 3.12.14
PyTorch: 2.11.0
Transformers: 5.17.0
NumPy: 1.26.4
Pandas: 2.3.3
CUDA: None
GPU: not available
Granite revision: b25344f89616dbcbdeaa5e4d9ecd40aab068e4da
GIFT-Eval revision: ce4ba193b086b80e440829961442eb05b693beb0


## Smoke test

Run the smoke test before starting the complete benchmark.

The smoke test evaluates `us_births/M` and verifies:

- GIFT-Eval data access
- Hugging Face model downloads
- Loading of all four ensemble members
- CUDA or CPU inference
- Ensemble aggregation
- Metric calculation
- Result and member-result generation

The smoke test writes to `outputs/granite_tsfm_ensemble/smoke` and does not modify the submitted leaderboard result.

In [7]:
smoke_settings = {
    **SETTINGS,
    "datasets": ["us_births/M"],
    "out_dir": output_root / "smoke",
}
smoke_csv = run_evaluation(**smoke_settings)
display(pd.read_csv(smoke_csv).sort_values("dataset"))

Found 1 already processed datasets. Skipping them.


Processing datasets: 100%|██████████| 1/1 [00:00<00:00, 9098.27it/s]


Skipping already processed dataset: us_births/M/short


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
0,us_births/M/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,3.844037e+07,3.844037e+07,4854.365062,0.549724,0.014852,0.015013,3.110399,6200.029562,0.019257,0.015078,0.01145,Healthcare,1


## Full GIFT-Eval benchmark

The full evaluation can take considerably longer than an interactive notebook session. A CUDA GPU and a batch or persistent compute session are recommended.

Set:

```python
RUN_FULL_BENCHMARK = True

Generated files are written beneath:

outputs/granite_tsfm_ensemble/full/
└── <leaderboard-model-name>/
    ├── all_results.csv
    ├── execution_log.csv
    └── members/
        └── <member-name>/
            └── all_results.csv



In [8]:
RUN_FULL_BENCHMARK = os.environ.get("RUN_FULL_BENCHMARK", "false").lower() in {"1", "true", "yes"}
full_csv = output_root / "full" / MODEL_NAME / SETTINGS["out_name"]
if RUN_FULL_BENCHMARK:
    full_settings = {
        **SETTINGS,
        "datasets": None,
        "out_dir": output_root / "full",
    }
    full_csv = run_evaluation(**full_settings)
else:
    print("Full evaluation not started.")

Full evaluation not started.


In [11]:
full_csv

PosixPath('/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/outputs/granite_tsfm_ensemble/full/probability-ensemble-uniform-ibm-tsfm-granite-pt/all_results.csv')


## Validate and compare results

After the full benchmark completes, the following checks are applied:

1. Every expected GIFT-Eval task is present exactly once.
2. The output follows the required leaderboard schema.
3. All evaluation metrics are finite.
4. The leaderboard model name is consistent across rows.
5. A complete result file exists for every ensemble member.
6. When a submitted result is available locally, the regenerated metrics are compared with it using a small numerical tolerance.

Minor floating-point differences can occur between compatible hardware and software environments. Larger or systematic differences should be investigated using the saved member results and execution log.


In [15]:
if full_csv.is_file():
    results = pd.read_csv(full_csv).sort_values("dataset").reset_index(drop=True)
    expected_rows = (
        len(experiment_config["datasets"]["short"])
        + 2 * len(experiment_config["datasets"]["medium_long"])
    )
    metrics = results.filter(regex=r"^eval_metrics/")
    assert len(results) == results["dataset"].nunique() == expected_rows
    assert results.shape[1] == 15
    assert results["model"].eq(MODEL_NAME).all()
    assert metrics.shape[1] == 11 and np.isfinite(metrics.to_numpy()).all()

    expected_members = experiment_config["configurations"][MODEL_NAME]["model_names"]

    for member_name in expected_members:
        member_csv = (
            full_csv.parent
            / "members"
            / member_name
            / SETTINGS["out_name"]
        )
        assert member_csv.is_file(), f"Missing member results: {member_name}"

        member_results = pd.read_csv(member_csv)
        assert not member_results.empty, (
            f"No member results produced for {member_name}"
        )
        assert member_results["dataset"].is_unique, (
            f"Duplicate member results found for {member_name}"
        )

    submitted_csv = gift_eval_root / "results" / MODEL_NAME / "all_results.csv"
    if submitted_csv.is_file():
        submitted = pd.read_csv(submitted_csv).sort_values("dataset").reset_index(drop=True)
        pd.testing.assert_frame_equal(
            results, submitted, check_dtype=False, rtol=1e-5, atol=1e-7
        )
        print(f"Generated metrics match {submitted_csv}")
    else:
        print(f"Copy the validated CSV to {submitted_csv} for submission.")
    display(results)
else:
    print("Run the full benchmark before validating results.")

Copy the validated CSV to /Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/results/probability-ensemble-uniform-ibm-tsfm-granite-pt/all_results.csv for submission.


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
0,bitbrains_fast_storage/5T/long,probability-ensemble-uniform-ibm-tsfm-granite-pt,5.167096e+06,5.167096e+06,415.198592,0.895427,5.113483,0.780921,15.335018,2273.124825,6.007190,1.097246,0.738636,Web/CloudOps,2
1,bitbrains_fast_storage/5T/medium,probability-ensemble-uniform-ibm-tsfm-granite-pt,2.979697e+06,2.979697e+06,264.582289,0.971445,3.129903,0.741148,20.969630,1726.179988,5.244612,0.803874,0.597573,Web/CloudOps,2
2,bitbrains_fast_storage/5T/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,1.818745e+06,1.818745e+06,157.937975,0.667918,1.774113,0.555590,13.245458,1348.608471,4.234297,0.495886,0.377641,Web/CloudOps,2
3,bitbrains_fast_storage/H/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,2.751093e+06,2.751093e+06,307.982551,0.943165,3.813678,0.500558,20.296691,1658.641980,4.727668,0.877850,0.642808,Web/CloudOps,2
4,bitbrains_rnd/5T/long,probability-ensemble-uniform-ibm-tsfm-granite-pt,2.315918e+06,2.315918e+06,181.295877,3.309184,2.089985,0.680030,115.533259,1521.814043,5.829727,0.694504,0.624268,Web/CloudOps,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,sz_taxi/H/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,7.119625e+00,7.119625e+00,1.849266,0.563207,1.257332,0.296337,4.403551,2.668263,0.248565,0.172271,0.136926,Transport,1
93,temperature_rain/D/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,1.860590e+02,1.860590e+02,5.809830,1.348255,20.769778,1.500023,23.459091,13.640344,1.605829,0.683971,0.559544,Nature,1
94,us_births/D/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,1.538509e+05,1.538509e+05,235.722596,0.347096,0.022746,0.022369,3.076063,392.238257,0.036769,0.022097,0.017692,Healthcare,1
95,us_births/M/short,probability-ensemble-uniform-ibm-tsfm-granite-pt,3.844037e+07,3.844037e+07,4854.365062,0.549724,0.014852,0.015013,3.110399,6200.029562,0.019257,0.015078,0.011450,Healthcare,1


In [16]:
full_csv

PosixPath('/Users/fearghalodonncha/Work/DigitalTwin/TSFM_Agent/Development/tsfm_agent_repos/gift-eval-fork/gift-eval/outputs/granite_tsfm_ensemble/full/probability-ensemble-uniform-ibm-tsfm-granite-pt/all_results.csv')